<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [4]:
using System;
using System.Collections.Generic;
using System.Linq;

public delegate void ProductEventHandler(Product product, string message);
public delegate void InventoryEventHandler<T>(T item, int quantity) where T : Product;
public delegate void StoreEventHandler(string message);

public interface IProductService
{
    void UpdatePrice(Product product, decimal newPrice);
    string GetProductDetails(Product product);
    bool ValidateProduct(Product product);
}

public interface IInventoryService
{
    void RestockProduct(Product product, int quantity);
    void ProcessReturn(Product product);
}

public class Product : IProductService, IInventoryService
{
    private string name;
    private decimal price;
    private int quantity;
    protected string category;
    internal string description;
    private string manufacturer;
    private string barcode;
    protected double weight;
    internal int popularity;
    private DateTime creationDate;
    protected bool isEcoFriendly;
    internal int warrantyMonths;
    private string countryOfOrigin;
    private List<string> tags;
    private Dictionary<string, string> specifications;
    private HashSet<string> compatibleProducts;
    private Queue<string> priceHistory;
    public event ProductEventHandler PriceChanged;
    public event ProductEventHandler LowStockWarning;
    public event ProductEventHandler ProductUpdated;

    public Product(string name, decimal price, int quantity, string category,
                  string description, string manufacturer, string barcode, double weight,
                  DateTime creationDate, bool isEcoFriendly, int warrantyMonths, string countryOfOrigin)
    {
        this.name = name;
        this.price = price;
        this.quantity = quantity;
        this.category = category;
        this.description = description;
        this.manufacturer = manufacturer;
        this.barcode = barcode;
        this.weight = weight;
        this.popularity = 0;
        this.creationDate = creationDate;
        this.isEcoFriendly = isEcoFriendly;
        this.warrantyMonths = warrantyMonths;
        this.countryOfOrigin = countryOfOrigin;
        this.tags = new List<string>();
        this.specifications = new Dictionary<string, string>();
        this.compatibleProducts = new HashSet<string>();
        this.priceHistory = new Queue<string>();
        priceHistory.Enqueue($"{DateTime.Now:yyyy-MM-dd}: {price:C}");
    }

    protected virtual void OnProductUpdated(string message)
    {
        ProductUpdated?.Invoke(this, message);
    }

    protected virtual void OnPriceChanged(string message)
    {
        PriceChanged?.Invoke(this, message);
    }

    protected virtual void OnLowStockWarning(string message)
    {
        LowStockWarning?.Invoke(this, message);
    }

    public void AddTag(string tag)
    {
        tags.Add(tag);
        OnProductUpdated($"Добавлен тег: {tag}");
    }

    public void AddSpecification(string key, string value)
    {
        specifications[key] = value;
        OnProductUpdated($"Добавлена спецификация: {key}");
    }

    public void AddCompatibleProduct(string productName)
    {
        compatibleProducts.Add(productName);
        OnProductUpdated($"Добавлен совместимый продукт: {productName}");
    }

    public List<string> GetTags() => new List<string>(tags);
    public Dictionary<string, string> GetSpecifications() => new Dictionary<string, string>(specifications);
    public HashSet<string> GetCompatibleProducts() => new HashSet<string>(compatibleProducts);

    public void DisplayPriceHistory()
    {
        Console.WriteLine($"История цен для {name}:");
        foreach (var history in priceHistory)
        {
            Console.WriteLine($"  {history}");
        }
    }

    public int GetProductAge()
    {
        return (DateTime.Now - creationDate).Days;
    }

    public virtual decimal CalculateEnvironmentalFee()
    {
        return isEcoFriendly ? 0 : (decimal)weight * 10;
    }

    public void ExtendWarranty(int additionalMonths)
    {
        warrantyMonths += additionalMonths;
        Console.WriteLine($"Гарантия продлена на {additionalMonths} месяцев");
        OnProductUpdated("Гарантия продлена");
    }

    public string GetOriginInfo()
    {
        return $"Страна: {countryOfOrigin} | Производитель: {manufacturer}";
    }

    public void IncreasePopularity() => popularity++;
    public double GetShippingCost() => weight * 0.5;
    public virtual string GetSpecialInfo() => $"Категория: {category}, Вес: {weight}кг";

    public void UpdateQuantity(int amount)
    {
        quantity += amount;
        if (quantity < 3)
        {
            OnLowStockWarning($"Низкий запас: осталось {quantity} единиц");
        }
    }

    public void UpdateQuantity(int amount, string reason)
    {
        quantity += amount;
        Console.WriteLine($"Изменение кол-ва ({reason}): {name}");
        
        if (quantity < 3)
        {
            OnLowStockWarning($"Низкий запас: осталось {quantity} единиц");
        }
    }

    public virtual void ApplyDiscount(decimal percent)
    {
        decimal oldPrice = price;
        price -= price * (percent / 100);
        priceHistory.Enqueue($"{DateTime.Now:yyyy-MM-dd}: {price:C} (скидка {percent}%)");
        Console.WriteLine($"Скидка {percent}% применена");
        
        OnPriceChanged($"Цена изменена с {oldPrice:C} на {price:C}");
    }

    public string GetProductInfo() => $"{name} | Цена: {price:C} | Кол-во: {quantity}";

    void IProductService.UpdatePrice(Product product, decimal newPrice)
    {
        if (product == this)
        {
            decimal oldPrice = price;
            price = newPrice;
            priceHistory.Enqueue($"{DateTime.Now:yyyy-MM-dd}: {newPrice:C}");
            Console.WriteLine($"Цена изменена с {oldPrice} на {newPrice}");
            
            OnPriceChanged($"Цена изменена с {oldPrice:C} на {newPrice:C}");
        }
    }

    string IProductService.GetProductDetails(Product product)
    {
        return product == this ? 
            $"{name} | {description} | Гарантия: {warrantyMonths} мес." : 
            "Продукт не найден";
    }

    bool IProductService.ValidateProduct(Product product)
    {
        return product == this && !string.IsNullOrEmpty(barcode) && quantity >= 0;
    }

    void IInventoryService.RestockProduct(Product product, int quantity)
    {
        if (product == this)
        {
            UpdateQuantity(quantity, "ресток");
            Console.WriteLine($"Продукт {name} пополнен на {quantity} единиц");
        }
    }

    void IInventoryService.ProcessReturn(Product product)
    {
        if (product == this)
        {
            UpdateQuantity(1, "возврат");
            Console.WriteLine($"Обработан возврат для {name}");
        }
    }
}

public class ElectronicProduct : Product
{
    private int powerConsumption;
    private bool hasBattery;
    protected int connectivityType;
    internal string model;
    private bool isRefurbished;
    protected string energyClass;
    internal int standbyPower;
    private string safetyCertification;
    private List<string> supportedStandards;
    private Dictionary<string, string> technicalSpecs;
    private HashSet<string> connectivityOptions;

    public ElectronicProduct(string name, decimal price, int quantity, string description,
                           string manufacturer, string barcode, double weight,
                           int powerConsumption, bool hasBattery, int connectivityType,
                           DateTime creationDate, bool isEcoFriendly, int warrantyMonths, string countryOfOrigin,
                           bool isRefurbished, string energyClass, int standbyPower, string safetyCertification)
        : base(name, price, quantity, "Электроника", description, manufacturer, barcode, weight,
              creationDate, isEcoFriendly, warrantyMonths, countryOfOrigin)
    {
        this.powerConsumption = powerConsumption;
        this.hasBattery = hasBattery;
        this.connectivityType = connectivityType;
        this.model = "Base Model";
        this.isRefurbished = isRefurbished;
        this.energyClass = energyClass;
        this.standbyPower = standbyPower;
        this.safetyCertification = safetyCertification;
        this.supportedStandards = new List<string>();
        this.technicalSpecs = new Dictionary<string, string>();
        this.connectivityOptions = new HashSet<string>();
    }

    public void AddSupportedStandard(string standard)
    {
        supportedStandards.Add(standard);
        OnProductUpdated($"Добавлен поддерживаемый стандарт: {standard}");
    }

    public void AddTechnicalSpec(string key, string value)
    {
        technicalSpecs[key] = value;
        OnProductUpdated($"Добавлена техническая спецификация: {key}");
    }

    public void AddConnectivityOption(string option)
    {
        connectivityOptions.Add(option);
        OnProductUpdated($"Добавлена опция подключения: {option}");
    }

    public List<string> GetSupportedStandards() => new List<string>(supportedStandards);
    public Dictionary<string, string> GetTechnicalSpecs() => new Dictionary<string, string>(technicalSpecs);
    public HashSet<string> GetConnectivityOptions() => new HashSet<string>(connectivityOptions);

    public bool IsEnergyEfficient()
    {
        return energyClass == "A" || energyClass == "A+";
    }

    public decimal CalculateAnnualEnergyCost(double hoursPerDay, decimal costPerKwh)
    {
        decimal dailyConsumption = (decimal)((powerConsumption * hoursPerDay + standbyPower * 24) / 1000.0);
        return dailyConsumption * 365 * costPerKwh;
    }

    public string GetCertificationInfo()
    {
        return $"Сертификация: {safetyCertification} | Энергокласс: {energyClass}";
    }

    public override decimal CalculateEnvironmentalFee()
    {
        decimal baseFee = base.CalculateEnvironmentalFee();
        return hasBattery ? baseFee + 500 : baseFee;
    }

    public bool CanConnectToWiFi() => connectivityType == 1;
    public void ChargeDevice() => Console.WriteLine("Зарядка устройства...");
    public override string GetSpecialInfo() => $"Потребление: {powerConsumption}Вт | Модель: {model}";

    public override void ApplyDiscount(decimal percent)
    {
        base.ApplyDiscount(percent);
        if (percent > 10) popularity += 5;
    }
}

public class Smartphone : ElectronicProduct
{
    private string os;
    private int storageGB;
    protected string color;
    internal bool is5GEnabled;
    private string processorModel;
    protected int batteryHealth;
    internal bool hasFaceRecognition;
    private int cameraCount;
    private List<string> installedApps;
    private Dictionary<DateTime, string> updateHistory;
    private HashSet<string> supportedNetworks;

    public Smartphone(string name, decimal price, int quantity, string description,
                     string manufacturer, string barcode, double weight,
                     int powerConsumption, bool hasBattery, int connectivityType,
                     string os, int storageGB, string color,
                     DateTime creationDate, bool isEcoFriendly, int warrantyMonths, string countryOfOrigin,
                     bool isRefurbished, string energyClass, int standbyPower, string safetyCertification,
                     string processorModel, int batteryHealth, bool hasFaceRecognition, int cameraCount)
        : base(name, price, quantity, description, manufacturer, barcode, weight,
              powerConsumption, hasBattery, connectivityType, creationDate, isEcoFriendly,
              warrantyMonths, countryOfOrigin, isRefurbished, energyClass, standbyPower, safetyCertification)
    {
        this.os = os;
        this.storageGB = storageGB;
        this.color = color;
        this.is5GEnabled = true;
        this.model = "Smartphone";
        this.processorModel = processorModel;
        this.batteryHealth = batteryHealth;
        this.hasFaceRecognition = hasFaceRecognition;
        this.cameraCount = cameraCount;
        this.installedApps = new List<string>();
        this.updateHistory = new Dictionary<DateTime, string>();
        this.supportedNetworks = new HashSet<string>();
    }

    public void InstallApplication(string appName)
    {
        installedApps.Add(appName);
        Console.WriteLine($"Приложение {appName} установлено");
    }

    public void RecordUpdate(string updateInfo)
    {
        updateHistory[DateTime.Now] = updateInfo;
        Console.WriteLine($"Записан апдейт: {updateInfo}");
    }

    public void AddSupportedNetwork(string network)
    {
        supportedNetworks.Add(network);
        Console.WriteLine($"Добавлена поддержка сети: {network}");
    }

    public void DisplayInstalledApps()
    {
        Console.WriteLine($"Установленные приложения ({installedApps.Count}):");
        foreach (var app in installedApps)
        {
            Console.WriteLine($"  - {app}");
        }
    }

    public void DisplayUpdateHistory()
    {
        Console.WriteLine("История обновлений:");
        foreach (var update in updateHistory.OrderBy(u => u.Key))
        {
            Console.WriteLine($"  {update.Key:yyyy-MM-dd}: {update.Value}");
        }
    }

    public void OptimizeBattery()
    {
        if (batteryHealth < 100)
        {
            batteryHealth += 5;
            Console.WriteLine($"Здоровье батареи улучшено до {batteryHealth}%");
        }
    }

    public string GetPerformanceInfo()
    {
        return $"Процессор: {processorModel} | Память: {storageGB}GB | Камеры: {cameraCount}";
    }

    public bool NeedsBatteryReplacement()
    {
        return batteryHealth < 80;
    }

    public void TakePortraitPhoto()
    {
        if (cameraCount >= 2)
            Console.WriteLine("Портретное фото сделано с эффектом боке");
        else
            Console.WriteLine("Недостаточно камер для портретного режима");
    }

    public void MakeCall() => Console.WriteLine("Вызов совершён");
    public void InstallApp(string appName) => Console.WriteLine($"Установка {appName}");
    public override string GetSpecialInfo() => $"{os} | Память: {storageGB}GB | Цвет: {color}";

    public void UpdateStorage(int newStorage) => storageGB = newStorage;
    public void UpdateStorage(int newStorage, bool cloudMigration)
    {
        storageGB = newStorage;
        if (cloudMigration) Console.WriteLine("Данные перенесены в облако");
    }
}

public class DependencyContainer
{
    private readonly Dictionary<Type, object> _services = new Dictionary<Type, object>();

    public void Register<T>(T service)
    {
        _services[typeof(T)] = service;
    }

    public T Resolve<T>()
    {
        if (_services.TryGetValue(typeof(T), out var service))
        {
            return (T)service;
        }
        throw new InvalidOperationException($"Сервис типа {typeof(T)} не зарегистрирован");
    }
}

public interface INotificationService
{
    void SendLowStockAlert(Product product);
    void SendPriceChangeNotification(Product product, decimal oldPrice);
}

public class EmailNotificationService : INotificationService
{
    public void SendLowStockAlert(Product product)
    {
        Console.WriteLine($"Низкий запас для {product.GetProductInfo()}");
    }

    public void SendPriceChangeNotification(Product product, decimal oldPrice)
    {
        Console.WriteLine($"Цена {product.GetProductInfo()} изменена с {oldPrice:C}");
    }
}

public interface IAnalyticsService
{
    void TrackProductView(Product product);
    void TrackSale(Product product, int quantity);
}

public class GoogleAnalyticsService : IAnalyticsService
{
    public void TrackProductView(Product product)
    {
        Console.WriteLine($"Просмотр товара {product.GetProductInfo()}");
    }

    public void TrackSale(Product product, int quantity)
    {
        Console.WriteLine($"Продано {quantity} единиц {product.GetProductInfo()}");
    }
}

public class Inventory<T> where T : Product
{
    private List<T> items = new List<T>();
    private Dictionary<string, T> itemsByBarcode = new Dictionary<string, T>();
    private readonly INotificationService _notificationService;
    private readonly IAnalyticsService _analyticsService;
    public event InventoryEventHandler<T> ItemAdded;
    public event InventoryEventHandler<T> ItemRemoved;
    public event InventoryEventHandler<T> LowStockDetected;

    public Inventory(INotificationService notificationService, IAnalyticsService analyticsService)
    {
        _notificationService = notificationService;
        _analyticsService = analyticsService;
    }

    public void AddItem(T item)
    {
        items.Add(item);
        itemsByBarcode[item.GetProductInfo().Split('|')[0].Trim()] = item;
        _analyticsService.TrackProductView(item);
        
        ItemAdded?.Invoke(item, items.Count);
        item.LowStockWarning += OnProductLowStock;
    }
    
    public void RemoveItem(T item)
    {
        items.Remove(item);
        itemsByBarcode.Remove(item.GetProductInfo().Split('|')[0].Trim());
        ItemRemoved?.Invoke(item, items.Count);
    }

    public T FindItemByBarcode(string barcode)
    {
        return items.FirstOrDefault(item => item.GetProductInfo().Contains(barcode));
    }

    public List<T> FindItemsByTag(string tag)
    {
        return items.Where(item => item.GetProductInfo().ToLower().Contains(tag.ToLower())).ToList();
    }

    public void PrintInventory()
    {
        Console.WriteLine($"\nИнвентарь ({typeof(T).Name}):");
        foreach (var item in items)
        {
            Console.WriteLine(item.GetProductInfo());
        }
    }

    public void CheckLowStock(int threshold = 5)
    {
        var lowStockItems = items.Where(item => item.GetProductInfo().Contains($"Кол-во: {threshold}")).ToList();
        foreach (var item in lowStockItems)
        {
            _notificationService.SendLowStockAlert(item);
            LowStockDetected?.Invoke(item, threshold);
        }
    }

    private void OnProductLowStock(Product product, string message)
    {
        Console.WriteLine($"ПРЕДУПРЕЖДЕНИЕ: {message}");
    }
}

public class Seller
{
    private string name;
    private List<Product> products = new List<Product>();
    private int rating;
    protected string specialization;
    internal int yearsExperience;
    private string contactEmail;
    protected string shopName;
    internal decimal totalRevenue;
    private int successfulSales;
    private Dictionary<string, int> productsSold;
    private HashSet<string> customerFeedback;
    private Queue<string> pendingOrders;
    public event StoreEventHandler NewSale;
    public event StoreEventHandler RatingChanged;

    public Seller(string name, string specialization, int yearsExperience, 
                 string contactEmail, string shopName)
    {
        this.name = name;
        this.specialization = specialization;
        this.yearsExperience = yearsExperience;
        this.rating = 5;
        this.contactEmail = contactEmail;
        this.shopName = shopName;
        this.totalRevenue = 0;
        this.successfulSales = 0;
        this.productsSold = new Dictionary<string, int>();
        this.customerFeedback = new HashSet<string>();
        this.pendingOrders = new Queue<string>();
    }

    public void RecordSale(string productName, int quantity)
    {
        if (productsSold.ContainsKey(productName))
            productsSold[productName] += quantity;
        else
            productsSold[productName] = quantity;
    }

    public void AddCustomerFeedback(string feedback)
    {
        customerFeedback.Add(feedback);
        Console.WriteLine($"Добавлен отзыв: {feedback}");
    }

    public void AddPendingOrder(string order)
    {
        pendingOrders.Enqueue(order);
        Console.WriteLine($"Добавлен заказ в очередь: {order}");
    }

    public void ProcessNextOrder()
    {
        if (pendingOrders.Count > 0)
        {
            var order = pendingOrders.Dequeue();
            Console.WriteLine($"Обработан заказ: {order}");
        }
    }

    public void DisplaySalesStatistics()
    {
        Console.WriteLine($"Статистика продаж для {name}:");
        foreach (var sale in productsSold)
        {
            Console.WriteLine($"  {sale.Key}: {sale.Value} шт.");
        }
    }

    public void AddRevenue(decimal amount)
    {
        totalRevenue += amount;
        successfulSales++;
    }

    public decimal GetAverageSale()
    {
        return successfulSales > 0 ? totalRevenue / successfulSales : 0;
    }

    public string GetContactInfo()
    {
        return $"{name} | Email: {contactEmail} | Магазин: {shopName}";
    }

    public bool IsPremiumSeller()
    {
        return rating >= 4.5m && successfulSales > 50;
    }

    public void IncreaseRating()
    {
        rating++;
        RatingChanged?.Invoke($"Рейтинг продавца {name} повышен до {rating}");
    }

    public bool IsExpert() => yearsExperience > 3;
    public string GetSellerStats() => $"{name} | Рейтинг: {rating} | Опыт: {yearsExperience} лет";

    public void AddProduct(Product product) => products.Add(product);
    
    public void SellProduct(Product product)
    {
        product.UpdateQuantity(-1, "продажа");
        IncreaseRating();
        AddRevenue(1000);
        RecordSale(product.GetProductInfo().Split('|')[0].Trim(), 1);
        NewSale?.Invoke($"Продажа: {product.GetProductInfo()}");
    }
}

public class Store
{
    private string storeName;
    private List<Seller> sellers = new List<Seller>();
    private string phoneNumber;
    protected string owner;
    internal int totalCustomers;
    private string address;
    protected DateTime openingDate;
    internal bool isOnlineStore;
    private string website;
    private Dictionary<string, Seller> sellersByName;
    private HashSet<string> storeLocations;
    private Queue<string> customerServiceRequests;
    public event StoreEventHandler StoreOpened;
    public event StoreEventHandler NewSellerJoined;
    public event StoreEventHandler CustomerVisited;

    public Store(string storeName, string phoneNumber, string owner,
                 string address, DateTime openingDate, bool isOnlineStore, string website)
    {
        this.storeName = storeName;
        this.phoneNumber = phoneNumber;
        this.owner = owner;
        this.totalCustomers = 0;
        this.address = address;
        this.openingDate = openingDate;
        this.isOnlineStore = isOnlineStore;
        this.website = website;
        this.sellersByName = new Dictionary<string, Seller>();
        this.storeLocations = new HashSet<string>();
        this.customerServiceRequests = new Queue<string>();
        storeLocations.Add(address);

        StoreOpened?.Invoke($"Магазин {storeName} открыт!");
    }

    public void AddStoreLocation(string location)
    {
        storeLocations.Add(location);
        Console.WriteLine($"Добавлена новая локация: {location}");
    }

    public void AddCustomerServiceRequest(string request)
    {
        customerServiceRequests.Enqueue(request);
        Console.WriteLine($"Добавлен запрос в службу поддержки: {request}");
    }

    public void ProcessNextServiceRequest()
    {
        if (customerServiceRequests.Count > 0)
        {
            var request = customerServiceRequests.Dequeue();
            Console.WriteLine($"Обработан запрос поддержки: {request}");
        }
    }

    public void DisplayAllLocations()
    {
        Console.WriteLine($"Локации магазина {storeName}:");
        foreach (var location in storeLocations)
        {
            Console.WriteLine($"  - {location}");
        }
    }

    public Seller FindSellerByName(string name)
    {
        return sellersByName.ContainsKey(name) ? sellersByName[name] : null;
    }

    public int GetStoreAge()
    {
        return (DateTime.Now - openingDate).Days / 365;
    }

    public string GetFullAddress()
    {
        return $"{address} | Тел: {phoneNumber}";
    }

    public void LaunchOnlineStore(string newWebsite)
    {
        website = newWebsite;
        isOnlineStore = true;
        Console.WriteLine($"Онлайн-магазин запущен: {website}");
    }

    public bool IsLongRunningBusiness()
    {
        return GetStoreAge() > 5;
    }

    public void AddCustomer()
    {
        totalCustomers++;
        CustomerVisited?.Invoke($"Новый клиент! Всего клиентов: {totalCustomers}");
    }

    public string GetStoreContacts() => $"Тел: {phoneNumber} | Сайт: {website}";
    public void ChangeOwner(string newOwner) => owner = newOwner;

    public void AddSeller(Seller seller)
    {
        sellers.Add(seller);
        sellersByName[seller.GetSellerStats().Split('|')[0].Trim()] = seller;
        NewSellerJoined?.Invoke($"Новый продавец: {seller.GetSellerStats()}");
        seller.NewSale += OnNewSale;
        seller.RatingChanged += OnSellerRatingChanged;
    }
    
    public void PrintStoreInfo()
    {
        Console.WriteLine($"\nМагазин: {storeName}");
        Console.WriteLine($"Владелец: {owner} | Клиентов: {totalCustomers}");
        Console.WriteLine($"Адрес: {address} | Онлайн: {isOnlineStore}");
    }

    private void OnNewSale(string message)
    {
        Console.WriteLine($"STORE NOTIFICATION: {message}");
    }

    private void OnSellerRatingChanged(string message)
    {
        Console.WriteLine($"STORE NOTIFICATION: {message}");
    }
}


        void OnPriceChanged(Product product, string message)
        {
            Console.WriteLine($"СОБЫТИЕ: {message}");
        }

        void OnLowStockWarning(Product product, string message)
        {
            Console.WriteLine($"СОБЫТИЕ: {message}");
        }

        void OnItemAdded(Product product, int quantity)
        {
            Console.WriteLine($"СОБЫТИЕ: Добавлен товар в инвентарь. Всего: {quantity}");
        }

        var container = new DependencyContainer();
        container.Register<INotificationService>(new EmailNotificationService());
        container.Register<IAnalyticsService>(new GoogleAnalyticsService());

        var smartphone = new Smartphone(
            "iPhone 15", 150000, 10, "Флагманский смартфон",
            "Apple", "123456", 0.2, 20, true, 1,
            "iOS", 256, "Чёрный",
            new DateTime(2024, 1, 15), true, 24, "США",
            false, "A+", 1, "FCC/CE",
            "A17 Pro", 95, true, 3
        );

        var book = new Product(
            "Война и мир", 1500, 5, "Книги",
            "Классика", "Эксмо", "789012", 0.5,
            new DateTime(2023, 12, 1), true, 0, "Россия"
);

smartphone.PriceChanged += OnPriceChanged;
smartphone.LowStockWarning += OnLowStockWarning;
book.PriceChanged += OnPriceChanged;
book.LowStockWarning += OnLowStockWarning;

var seller = new Seller("Иван Петров", "Электроника", 5, 
                        "ivan@example.com", "ТехноМаркет");
        
var store = new Store("ТехноМир", "+79990001122", "Петр Иванов",
                        "ул. Пушкина, 25", new DateTime(2020, 5, 10), 
                        true, "technomir.ru");

store.StoreOpened += (msg) => Console.WriteLine($"{msg}");
store.NewSellerJoined += (msg) => Console.WriteLine($"{msg}");
store.CustomerVisited += (msg) => Console.WriteLine($"{msg}");

var electronicInventory = new Inventory<ElectronicProduct>(
    container.Resolve<INotificationService>(),
    container.Resolve<IAnalyticsService>()
);

electronicInventory.ItemAdded += OnItemAdded;
electronicInventory.ItemRemoved += (product, quantity) => 
    Console.WriteLine($"СОБЫТИЕ: Удален товар из инвентаря. Осталось: {quantity}");
electronicInventory.LowStockDetected += (product, threshold) => 
    Console.WriteLine($"СОБЫТИЕ: Низкий запас товара (порог: {threshold})");

electronicInventory.AddItem(smartphone);

store.AddSeller(seller);
seller.AddProduct(smartphone);
seller.AddProduct(book);

Console.WriteLine("=== Демонстрация новых атрибутов и методов ===");
        
store.PrintStoreInfo();
electronicInventory.PrintInventory();

Console.WriteLine($"\nВозраст продукта: {smartphone.GetProductAge()} дней");
Console.WriteLine($"Экологический сбор: {smartphone.CalculateEnvironmentalFee()}");

Console.WriteLine("\n=== Работа с коллекциями ===");
smartphone.AddTag("флагман");
smartphone.AddTag("смартфон");
smartphone.AddSpecification("Диагональ", "6.1\"");
smartphone.AddCompatibleProduct("AirPods Pro");
        
smartphone.InstallApplication("Telegram");
smartphone.InstallApplication("WhatsApp");
smartphone.RecordUpdate("iOS 17.2");
smartphone.AddSupportedNetwork("5G");
        
seller.RecordSale("iPhone 15", 3);
seller.AddCustomerFeedback("Отличный продавец!");
seller.AddPendingOrder("Заказ #001");
        
store.AddStoreLocation("ТЦ Мега");
store.AddCustomerServiceRequest("Консультация по iPhone");

smartphone.DisplayInstalledApps();
smartphone.DisplayUpdateHistory();
seller.DisplaySalesStatistics();
store.DisplayAllLocations();

Console.WriteLine("\n=== Демонстрация событий ===");
store.AddCustomer();
store.AddCustomer();
        
smartphone.ApplyDiscount(10);
book.UpdateQuantity(-4);

Console.WriteLine("\n=== Явная реализация интерфейса ===");
IProductService productService = smartphone;
productService.UpdatePrice(smartphone, 140000);
Console.WriteLine(productService.GetProductDetails(smartphone));

IInventoryService inventoryService = smartphone;
inventoryService.RestockProduct(smartphone, 5);

Console.WriteLine("\n=== Новые методы ===");
smartphone.OptimizeBattery();
smartphone.TakePortraitPhoto();
Console.WriteLine(smartphone.GetPerformanceInfo());
Console.WriteLine(seller.GetContactInfo());
Console.WriteLine(store.GetFullAddress());

electronicInventory.CheckLowStock();

Console.WriteLine("\n=== Статистика продавца ===");
Console.WriteLine($"Общая выручка: {seller.totalRevenue}");
Console.WriteLine($"Премиум-продавец: {seller.IsPremiumSeller()}");

Console.WriteLine("\n=== Информация о магазине ===");
Console.WriteLine($"Возраст бизнеса: {store.GetStoreAge()} лет");
Console.WriteLine($"Долгосрочный бизнес: {store.IsLongRunningBusiness()}");

Console.WriteLine("\n=== История цен ===");
smartphone.DisplayPriceHistory();

Просмотр товара iPhone 15 | Цена: 150 000,00 ¤ | Кол-во: 10
СОБЫТИЕ: Добавлен товар в инвентарь. Всего: 1
Новый продавец: Иван Петров | Рейтинг: 5 | Опыт: 5 лет
=== Демонстрация новых атрибутов и методов ===

Магазин: ТехноМир
Владелец: Петр Иванов | Клиентов: 0
Адрес: ул. Пушкина, 25 | Онлайн: True

Инвентарь (ElectronicProduct):
iPhone 15 | Цена: 150 000,00 ¤ | Кол-во: 10

Возраст продукта: 699 дней
Экологический сбор: 500

=== Работа с коллекциями ===
Приложение Telegram установлено
Приложение WhatsApp установлено
Записан апдейт: iOS 17.2
Добавлена поддержка сети: 5G
Добавлен отзыв: Отличный продавец!
Добавлен заказ в очередь: Заказ #001
Добавлена новая локация: ТЦ Мега
Добавлен запрос в службу поддержки: Консультация по iPhone
Установленные приложения (2):
  - Telegram
  - WhatsApp
История обновлений:
  2025-12-14: iOS 17.2
Статистика продаж для Иван Петров:
  iPhone 15: 3 шт.
Локации магазина ТехноМир:
  - ул. Пушкина, 25
  - ТЦ Мега

=== Демонстрация событий ===
Новый клиент! Все